# Reading the QuakeScope pick catalogue

Machine-learning phase picks for the western United States, as Parquet on S3.
**No credentials, no database, no account** — the catalogue is public-read and
every cell below runs anonymously.

This notebook:

1. installs what it needs (about a minute on a cold Colab runtime),
2. reads picks straight from S3 with partition pruning,
3. plots what is in them,
4. re-queries the **original waveforms** from FDSN with ObsPy and draws the
   picks on top, so you can see what the model saw.

| | |
|---|---|
| bucket | `s3://quakescope-picks-2026` (us-east-2, public-read) |
| layout | `<campaign>/picks/network=<NET>/year=<YYYY>/month=<MM>/*.parquet` |
| provenance | `<campaign>/runs/<rid>.json` — model, weight, thresholds |

Nothing here writes to S3, and nothing needs an AWS account.

## 1. Install

Safe to re-run; skip it if you already have these.

In [ ]:
# Colab already has pandas/numpy/matplotlib. These are the rest.
%pip install -q pyarrow s3fs obspy
print("ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BUCKET = "quakescope-picks-2026"
ANON = {"anon": True}          # public-read: no keys, no config, no account

## 2. Read some picks

The catalogue is Hive-partitioned by **network / year / month**. Pass `filters`
and only the matching files are fetched — the partition columns are read from
the *paths*, so an unfiltered read of the whole catalogue is never necessary.

Start with one month of the Southern California network `CI`.

In [ ]:
picks = pd.read_parquet(
    f"s3://{BUCKET}/scedc/picks/",
    filters=[("network", "=", "CI"), ("year", "=", 2014), ("month", "=", 9)],
    storage_options=ANON,
)
print(f"{len(picks):,} picks")
picks.head()

### What the columns mean

| column | meaning |
|---|---|
| `tid` | trace id, `NET.STA.LOC` |
| `cha` | channel band (`HH`, `BH`, `EH`, `HN`) — one band per station-day |
| `pha` | `P` or `S` |
| `start`, `peak`, `end` | pick window; **`peak` is the arrival time** |
| `conf` | model confidence, 0–1. Everything stored is above 0.2 |
| `amp` | Wood-Anderson amplitude, metres — for local magnitude |
| `amp_raw` | peak amplitude in raw counts, before response removal |
| `rid` | run id → `runs/<rid>.json` for the model and thresholds used |

`conf` is a **detection score, not a probability of correctness**. The stored
floor of 0.2 is deliberately permissive so you can choose your own threshold;
most analyses want considerably more than that.

In [ ]:
print(picks["pha"].value_counts().to_string(), "\n")
print(picks["conf"].describe().round(3).to_string(), "\n")
print(f"stations : {picks['tid'].nunique():,}")
print(f"bands    : {picks['cha'].value_counts().to_dict()}")
print(f"amp set  : {picks['amp'].notna().mean():.1%}")

## 3. What the month looks like

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

daily = picks.groupby([picks["peak"].dt.floor("D"), "pha"]).size().unstack(fill_value=0)
for ph, c in (("P", "#c1272d"), ("S", "#0b6e99")):
    ax[0].plot(daily.index, daily[ph], lw=1.2, color=c, label=ph)
ax[0].legend(title="phase")
ax[0].set_title("picks per day"); ax[0].set_ylabel("picks")
ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax[0].xaxis.set_major_locator(mdates.DayLocator(interval=7))

for ph, c in (("P", "#c1272d"), ("S", "#0b6e99")):
    ax[1].hist(picks.loc[picks.pha == ph, "conf"], bins=60, histtype="step",
               lw=1.3, color=c, label=ph)
ax[1].set_yscale("log"); ax[1].set_title("confidence"); ax[1].legend()

amp = picks["amp"].dropna()
amp = amp[amp > 0]
ax[2].hist(np.log10(amp), bins=60, color="#555")
ax[2].set_title("log10 Wood-Anderson amplitude (m)")

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

The daily curve is the useful sanity check: **P and S should track each other**,
and a day where one collapses is usually a data outage rather than a quiet
Earth. The spikes are real sequences.

## 4. Re-query the waveforms and check the picks

Picks are only as good as they look on the record. The catalogue stores no
waveforms, so fetch them from the **SCEDC FDSN service** with ObsPy — the same
data the picker read, from the archive of record.

Take the busiest station-minute in the month.

In [ ]:
import obspy
from obspy.clients.fdsn import Client

picks["minute"] = picks["peak"].dt.floor("min")
tid, minute = picks.groupby(["tid", "minute"]).size().idxmax()
net, sta, loc = tid.split(".")
print(f"{tid}  {minute}")

t0 = obspy.UTCDateTime(minute) - 10
st = Client("SCEDC", timeout=60).get_waveforms(
    network=net, station=sta, location=loc or "*", channel="HH?",
    starttime=t0, endtime=t0 + 80,
)
st.detrend("demean").filter("bandpass", freqmin=2, freqmax=20)
print(st)

In [ ]:
window = picks[(picks.tid == tid) &
               (picks.peak >= minute) &
               (picks.peak < minute + pd.Timedelta(minutes=1))]

fig, axes = plt.subplots(len(st), 1, figsize=(12, 6), sharex=True)
for ax, tr in zip(np.atleast_1d(axes), st):
    t = tr.times() + (tr.stats.starttime - t0)
    ax.plot(t, tr.data / np.abs(tr.data).max(), lw=0.45, color="#333")
    for _, p in window.iterrows():
        ax.axvline(obspy.UTCDateTime(p["peak"]) - t0, lw=1.3, alpha=0.85,
                   color="#c1272d" if p["pha"] == "P" else "#0b6e99")
    ax.set_ylabel(tr.stats.channel)
    ax.spines[["top", "right"]].set_visible(False)
axes[-1].set_xlabel(f"seconds after {t0.isoformat()}")
axes[0].set_title(f"{tid} — red P, blue S ({len(window)} picks)")
fig.tight_layout()

Zoom in on a single pick to judge it properly. Change `i` to step through them —
low-`conf` picks are where the interesting disagreements live.

In [ ]:
i = 0
p = window.sort_values("peak").iloc[i]
tp = obspy.UTCDateTime(p["peak"])
tr = st.select(component="Z")[0].slice(tp - 3, tp + 7)

plt.figure(figsize=(11, 2.8))
plt.plot(tr.times() - 3, tr.data, lw=0.7, color="#333")
plt.axvline(0, color="#c1272d" if p["pha"] == "P" else "#0b6e99", lw=1.5)
plt.title(f"{tid} {p['pha']}  conf={p['conf']:.3f}  {p['peak']}")
plt.xlabel("seconds from pick"); plt.gca().spines[["top","right"]].set_visible(False)
plt.tight_layout()

## 5. Provenance

Every pick carries a run id. The run record says exactly which model and
thresholds produced it — worth quoting in a paper, and worth checking before
comparing picks across campaigns.

In [ ]:
import json, urllib.request

rid = picks["rid"].iloc[0]
url = f"https://{BUCKET}.s3.us-east-2.amazonaws.com/scedc/runs/{rid}.json"
print(json.dumps(json.load(urllib.request.urlopen(url)), indent=2))

## 6. Going further

**Other campaigns.** Swap `scedc` in the path for `ncedc`, `earthscope`,
`western` or `obs`. Coverage differs — see the
[dashboard](https://seisscoped.github.io/QuakeScope/campaign_dashboard.html)
for what has been written so far.

**Bigger than memory.** DuckDB will push filters down and never materialise the
whole catalogue:

```python
import duckdb
duckdb.sql("INSTALL httpfs; LOAD httpfs; SET s3_region='us-east-2';")
duckdb.sql(f'''
    SELECT tid, count(*) n, avg(conf) conf
    FROM read_parquet('s3://quakescope-picks-2026/scedc/picks/**/*.parquet',
                      hive_partitioning=1)
    WHERE network='CI' AND year=2014
    GROUP BY tid ORDER BY n DESC LIMIT 20
''').df()
```

**Choosing a threshold.** Everything above `conf` 0.2 is stored, which is far
more permissive than most uses want. Raise it and check the effect on your own
target events rather than adopting a number from elsewhere.

**Associating into events.** These are per-station picks, not locations. A
phase associator (GaMMA, PyOcto) turns them into events.

**Amplitudes.** `amp` is Wood-Anderson displacement in metres, ready for local
magnitude. `amp_raw` is pre-response counts, useful mainly for QC.

---

Catalogue and code: [SeisSCOPED/QuakeScope](https://github.com/SeisSCOPED/QuakeScope)